In [7]:
import cv2
import mediapipe as mp
import csv
import os

# 1. ตั้งค่า MediaPipe Holistic
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

csv_filename = "asl_holistic_dataset.csv"

# สร้าง Header สำหรับ CSV (144 ค่า)
if not os.path.exists(csv_filename):
    with open(csv_filename, mode='w', newline='') as f:
        writer = csv.writer(f)
        header = ['label']
        # Left Hand (21 pts), Right Hand (21 pts), Arms: shoulders, elbows, wrists (6 pts)
        for i in range(21):
            header.extend([f'lh_pt{i}_x', f'lh_pt{i}_y', f'lh_pt{i}_z'])
        for i in range(21):
            header.extend([f'rh_pt{i}_x', f'rh_pt{i}_y', f'rh_pt{i}_z'])
        for i in range(6):
            header.extend([f'arm_pt{i}_x', f'arm_pt{i}_y', f'arm_pt{i}_z'])
        writer.writerow(header)

def extract_holistic_features(results):
    features = []
    
    # 1. มือซ้าย (21 จุด x 3 = 63 ค่า)
    if results.left_hand_landmarks:
        wrist = results.left_hand_landmarks.landmark[0]
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x - wrist.x, lm.y - wrist.y, lm.z - wrist.z])
    else:
        features.extend([0.0] * 63)
        
    # 2. มือขวา (21 จุด x 3 = 63 ค่า)
    if results.right_hand_landmarks:
        wrist = results.right_hand_landmarks.landmark[0]
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x - wrist.x, lm.y - wrist.y, lm.z - wrist.z])
    else:
        features.extend([0.0] * 63)
        
    # 3. แขนและไหล่ (จุด 11, 12, 13, 14, 15, 16 ของ Pose = 6 จุด x 3 = 18 ค่า)
    if results.pose_landmarks:
        # ใช้ไหล่ซ้าย (จุด 11) เป็นจุดอ้างอิงของแขน
        ref_pt = results.pose_landmarks.landmark[11]
        arm_indices = [11, 12, 13, 14, 15, 16] # Left/Right Shoulder, Elbow, Wrist
        for idx in arm_indices:
            lm = results.pose_landmarks.landmark[idx]
            features.extend([lm.x - ref_pt.x, lm.y - ref_pt.y, lm.z - ref_pt.z])
    else:
        features.extend([0.0] * 18)
        
    return features

if 'cap' in locals() and cap.isOpened(): cap.release()
cv2.destroyAllWindows()

cap = cv2.VideoCapture(0)
print("=== ระบบบันทึกพิกัด 2 มือ + แขน ===")
print("วิธีใช้: ทำท่าค้างไว้แล้วกดปุ่มอักษร (A-T) ค้างไว้ | กด 'ESC' เพื่อปิด")

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        frame = cv2.flip(frame, 1)
        h, w, c = frame.shape
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image_rgb)

        # วาดเส้นจุดต่อ
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

        key = cv2.waitKey(1) & 0xFF
        if key == 27: break # กด ESC เพื่อปิด
        pressed_char = chr(key).upper() if (97 <= key <= 122 or 65 <= key <= 90) else None

        if pressed_char:
            features = extract_holistic_features(results)
            with open(csv_filename, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([pressed_char] + features)
            
            cv2.putText(frame, f"SAVING: {pressed_char}", (20, h - 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        cv2.imshow('Holistic Data Collector', frame)

cap.release()
cv2.destroyAllWindows()

=== ระบบบันทึกพิกัด 2 มือ + แขน ===
วิธีใช้: ทำท่าค้างไว้แล้วกดปุ่มอักษร (A-T) ค้างไว้ | กด 'ESC' เพื่อปิด


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

dataset_path = 'asl_holistic_dataset.csv'
data = pd.read_csv(dataset_path)
data.dropna(inplace=True)

print(f"📦 Loaded data: {len(data)} rows")

X = data.drop('label', axis=1).values 
y = data['label'].values              

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("🧠 Training AI Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 ความแม่นยำของ AI ตัวนี้อยู่ที่: {accuracy * 100:.2f}%")

with open('asl_holistic_model.p', 'wb') as f:
    pickle.dump({'model': model}, f)

print("💾 Saved model to 'asl_holistic_model.p' successfully!")

📦 Loaded data: 711 rows
🧠 Training AI Model...
🎯 ความแม่นยำของ AI ตัวนี้อยู่ที่: 97.90%
💾 Saved model to 'asl_holistic_model.p' successfully!


In [1]:
import cv2
import mediapipe as mp
import pickle
import numpy as np

if 'cap' in locals() and cap.isOpened(): cap.release()
cv2.destroyAllWindows()

WORD_MAP = {
    'A': 'Hello',     'B': 'How',       'C': 'You',       'D': 'Your',
    'E': 'Name',      'F': 'What',      'G': 'My',        'H': "I'm",
    'I': 'Fine',      'J': 'Meet you',  'K': 'Nice',      'L': 'Thanks',
    'M': 'I Love You','N': 'COOL!',     'O': 'Again',     'P': 'Yes',
    'Q': 'No',        'R': 'Good',      'S': 'Bad',       'T': 'Morning', 'U': 'idle'
}

with open('asl_holistic_model.p', 'rb') as f:
    model_data = pickle.load(f)
    model = model_data['model']

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def extract_holistic_features(results):
    features = []
    if results.left_hand_landmarks:
        wrist = results.left_hand_landmarks.landmark[0]
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x - wrist.x, lm.y - wrist.y, lm.z - wrist.z])
    else:
        features.extend([0.0] * 63)
        
    if results.right_hand_landmarks:
        wrist = results.right_hand_landmarks.landmark[0]
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x - wrist.x, lm.y - wrist.y, lm.z - wrist.z])
    else:
        features.extend([0.0] * 63)
        
    if results.pose_landmarks:
        ref_pt = results.pose_landmarks.landmark[11]
        arm_indices = [11, 12, 13, 14, 15, 16]
        for idx in arm_indices:
            lm = results.pose_landmarks.landmark[idx]
            features.extend([lm.x - ref_pt.x, lm.y - ref_pt.y, lm.z - ref_pt.z])
    else:
        features.extend([0.0] * 18)
        
    return features

cap = cv2.VideoCapture(0)
print("🤖 ระบบแปลภาษามือแบบ 2 มือ + แขน เปิดใช้งานแล้ว! (กด 'q' เพื่อปิด)")

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        frame = cv2.flip(frame, 1)
        h, w, c = frame.shape
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image_rgb)

        # วาด Landmark บนหน้าจอ
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

        # ดึง Feature
        features = extract_holistic_features(results)
        
        # ทำนายผลเฉพาะเวลาพบมือหรือแขนในกล้อง
        if any(features):
            prediction = model.predict([features])
            predicted_letter = str(prediction[0]).upper()
            predicted_word = WORD_MAP.get(predicted_letter, predicted_letter)

            # แสดงผลคำแปลบนแถบสีส้มด้านบน
            cv2.rectangle(frame, (0, 0), (640, 50), (245, 117, 16), -1)
            cv2.putText(frame, f"Translation: {predicted_word}", (15, 35), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow('2-Hands & Arms Real-time Translator', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

c:\Users\epryw\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\epryw\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


🤖 ระบบแปลภาษามือแบบ 2 มือ + แขน เปิดใช้งานแล้ว! (กด 'q' เพื่อปิด)
